# Act 1 — Shared responsibility: who owns what

Before you can talk about observability or compliance, you need to know whose job it is. AWS owns the physical infrastructure and the layers it manages on your behalf. You own your data, your IAM configuration, and the parts of the stack the service does not handle for you.

The line shifts depending on the service. For EC2 you own most of the stack. For S3 you own data and policies but nothing else. For Lambda, AWS handles the runtime entirely. This act establishes the boundary that the rest of the notebook works from.

## Three observability services, plus a compliance layer

Three services answer three different questions about a running AWS environment. **CloudWatch** answers *how is it performing?* — metrics, logs, alarms, dashboards. **CloudTrail** answers *who did what?* — every API call across the account, recorded and queryable. **AWS Config** answers *what changed, and is it compliant?* — a timeline of every resource's configuration with rules that fire when anything drifts.

Underneath those three sits the **Shared Responsibility Model** — the line between what AWS secures and what's left for the customer to configure. And on top of them sits the compliance layer: **AWS Artifact** as the self-serve portal for AWS's own audit reports, and **Audit Manager** as the tool that collects evidence for *your* audits.

## The Shared Responsibility Model

The boundary between AWS's responsibility and the customer's is the most quoted security framing on the platform, and most architecture decisions live on top of it. AWS is responsible for **security *of* the cloud** — physical data centres, hardware, the hypervisor, the global network, the managed-service control planes. The customer is responsible for **security *in* the cloud** — data, identity, application code, and however they configure the services they use.

The line shifts depending on the service type. As you move from IaaS toward serverless, AWS takes on more of the stack.

| Responsibility | EC2 (IaaS) | RDS (managed) | S3 / Lambda (serverless) |
|---|---|---|---|
| Physical, hypervisor, hardware | AWS | AWS | AWS |
| OS patching | **customer** | AWS | AWS |
| Database / runtime patching | **customer** | AWS | AWS |
| Network firewall (Security Group) | **customer** | **customer** | n/a |
| **Data, IAM, configuration** | **customer** | **customer** | **customer** |

The last row never moves. **The customer is always responsible for their data, their IAM configuration, and how they use the service.** A public S3 bucket with private data in it is a customer problem, regardless of how many compliance certifications AWS holds.

The four corners worth holding in your head.

- **EC2** — maximum customer responsibility. OS install and patching, runtime, app code, security groups, IAM, encryption — all yours.
- **RDS** — AWS handles OS, the DB engine, Multi-AZ replication, backups. You still own users, schema, security group rules, encryption at rest (enable at creation), and network placement.
- **S3** — AWS owns the storage tier completely. You own bucket policies, public-access settings, versioning, encryption mode, replication config.
- **Lambda** — AWS owns the runtime, scaling, hardware. You own the function code, execution role, environment variables (and how you store secrets), VPC config.

The pattern: the closer you get to serverless, the more AWS handles — but **data and IAM are always yours**.

# Act 2 — CloudWatch: how is it doing?

With the responsibility model in hand, the first question about a running system is "how is it performing?" Is the CPU high? Are requests slow? Are errors climbing? Did something just break?

**CloudWatch** is the answer. Metrics, logs, alarms, dashboards — one place to watch behaviour. This act covers each piece in turn, plus the agent that ships in-OS data the hypervisor cannot see, and the deeper Container Insights and Lambda Insights variants for ECS, EKS, and Lambda.

## CloudWatch Metrics

A **metric** is a time-series of numeric data points — CPU percent, request count, error rate, latency. Metrics are the foundation of CloudWatch, and most of what follows (alarms, dashboards) is built on them.

Every metric has:

- **Namespace** — a logical grouping. AWS services emit into `AWS/EC2`, `AWS/RDS`, `AWS/Lambda`. Custom metrics use any namespace you like.
- **Metric name** — `CPUUtilization`, `Invocations`, `Duration`, `OrdersProcessed`.
- **Dimensions** — key/value pairs that identify the specific resource: `{InstanceId: i-abc123}`, `{FunctionName: validate-order}`. Each unique combination of dimensions is a separate metric stream.
- **Resolution** — standard is one-minute granularity; high-resolution custom metrics go down to one second.

### Standard vs custom

AWS services push **standard metrics** for free — EC2 CPU/network/disk I/O, S3 bucket size, Lambda invocation count and duration, ALB request counts. The catch worth remembering on EC2: standard metrics come from the **hypervisor**, so **memory utilisation and in-OS disk space are not collected by default**. Those require the **CloudWatch Agent** running inside the instance.

**Custom metrics** are anything you push yourself via `PutMetricData` — application counters, business KPIs, in-OS metrics from the Agent. The first 10,000 are free; after that there's a per-metric monthly charge.

### Retention

CloudWatch keeps metric data points longer the older they are, by aggregating to coarser resolution:

- Sub-minute resolution → 3 hours
- 1-minute → 15 days
- 5-minute → 63 days
- 1-hour → 15 months

For longer retention, export to S3 or stream metrics elsewhere — CloudWatch isn't a long-term data warehouse.

## CloudWatch Logs

Logs are the unstructured cousin of metrics — text lines from services and applications, collected and queryable. The hierarchy:

- **Log Group** — the container; one per logical source, with a retention setting (1 day to 10 years, or never expire).
- **Log Stream** — events from a single emitter (one Lambda execution environment, one EC2 instance, one container).
- **Log Events** — the actual lines, with timestamps.

Lambda automatically writes to `/aws/lambda/<function-name>`. ECS captures container stdout/stderr. API Gateway has access logs. CloudTrail can deliver trails to Logs. VPC Flow Logs, Route 53 query logs, and the CloudWatch Agent all land here too.

### Metric filters

A **metric filter** scans incoming log events for a pattern and increments a CloudWatch metric when it matches. The standard pattern: a filter like `?ERROR ?Exception` on an application log group emits an `ApplicationErrors` metric, and an alarm on that metric pages you. This is how you turn unstructured logs into actionable signals.

### CloudWatch Logs Insights

Logs Insights is a SQL-like query language over log groups. A typical query:

```
fields @timestamp, @message
| filter @message like /ERROR/
| stats count(*) by bin(5m)
| sort @timestamp desc
```

It runs across multiple log groups simultaneously, supports time-binning, statistical aggregations, parse expressions for structured fields, and renders results as tables or time-series charts. The right tool for ad-hoc investigation — "what happened around 03:42 last night" — without exporting logs to S3 and running Athena.

### Export and subscriptions

- **S3 export** — batch export of a log group to S3 for long-term archive or Athena queries; not real-time.
- **Subscription filter** — real-time streaming of log events that match a filter pattern, into **Kinesis Data Streams**, **Kinesis Firehose**, or **Lambda**. The standard plumbing for shipping logs to OpenSearch, Splunk, Datadog, or a custom processor.

Logs are encrypted at rest — KMS via your CMK if you want full control of decryption auditing.

## CloudWatch Alarms

An alarm watches a single metric (or a metric math expression) and transitions between three states based on how the metric compares to a threshold.

| State | Meaning |
|---|---|
| **OK** | within threshold |
| **ALARM** | breached threshold per the evaluation rules |
| **INSUFFICIENT_DATA** | not enough data points to evaluate — startup, gaps, paused service |

The evaluation parameters matter and trip people up: a five-minute **period**, three **evaluation periods**, and `2 out of 3` **datapoints to alarm** means *ALARM when the metric breached the threshold in at least 2 of the most recent 3 five-minute windows*. Setting these tighter than necessary causes flapping; setting them looser delays response.

### Actions

- **SNS notification** — the universal output: email, SMS, Lambda, fan out wherever.
- **Auto Scaling action** — scale an ASG in or out (the standard target-tracking and step-scaling backend).
- **EC2 action** — stop, terminate, reboot, or **recover** the instance (recover replaces the underlying host on hardware failure while preserving the instance ID and EBS volumes).
- **Systems Manager action** — kick off an SSM Automation runbook for self-healing.

### Composite alarms

Real incidents rarely match a single metric. A **composite alarm** combines child alarms with AND / OR / NOT logic — "ALARM if HighCPU **AND** HighMemory", "ALARM if any of the five regional health alarms is in ALARM". The big payoff is alert noise reduction: page on the composite, suppress the noisy children.

### Billing alarm

A billing alarm watches the `AWS/Billing` `EstimatedCharges` metric. Two specifics: **billing metrics only exist in `us-east-1`** regardless of where you operate, and you have to enable billing alerts in account preferences before the metric appears. The standard guardrail for any account: a billing alarm wired to SNS.

## CloudWatch Agent and Insights Variants

The **CloudWatch Agent** runs inside an EC2 instance (or on-premises host) and ships what the hypervisor can't see — **in-OS memory utilisation, swap, disk space, disk I/O at the filesystem level** — plus **log files** the OS or applications write. Configuration is JSON, often stored in SSM Parameter Store and pulled by the agent at startup. The instance's IAM role needs the `CloudWatchAgentServerPolicy` managed policy.

The agent is the answer to two common scenarios. "EC2 ran out of memory and nobody noticed" — install the agent, push memory metrics, alarm on them. "We need application logs in CloudWatch" — point the agent at the log files; it streams them into log groups.

Two CloudWatch variants worth knowing.

- **Container Insights** — enhanced metrics and logs for **ECS** and **EKS**. Per-container CPU, memory, network, disk; collects container stdout/stderr to log groups. Enable once per cluster; the data appears in dedicated dashboards.
- **Lambda Insights** — function-level diagnostics: cold start counts, init duration, memory utilisation curves, CPU time, network I/O per invocation. Delivered via a Lambda **layer** added to the function. Standard CloudWatch metrics for Lambda (invocations, errors, duration) work without it; Insights is the deeper view when those aren't enough.

In [ ]:
import boto3, time
from datetime import datetime, timedelta

cw   = boto3.client("cloudwatch", region_name="us-east-1")
logs = boto3.client("logs",       region_name="us-east-1")

# Publish a custom metric with dimensions
cw.put_metric_data(
    Namespace="MyApp/Business",
    MetricData=[{
        "MetricName": "OrdersProcessed",
        "Value": 142, "Unit": "Count",
        "Dimensions": [{"Name": "Environment", "Value": "production"}],
    }],
)

# Alarm: CPU > 80% in 2 of the last 3 five-minute windows, scale out + page ops
cw.put_metric_alarm(
    AlarmName="HighCPU-prod-web",
    Namespace="AWS/EC2", MetricName="CPUUtilization", Statistic="Average",
    Dimensions=[{"Name": "AutoScalingGroupName", "Value": "prod-web-asg"}],
    Period=300, EvaluationPeriods=3, DatapointsToAlarm=2,
    Threshold=80.0, ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    AlarmActions=[
        "arn:aws:sns:us-east-1:111122223333:ops-alerts",
        "arn:aws:autoscaling:us-east-1:111122223333:scalingPolicy:abc:autoScalingGroupName/prod-web-asg:policyName/scale-out",
    ],
)

# Logs Insights — count errors per 5-min bin in the last hour
q = logs.start_query(
    logGroupName="/aws/lambda/my-function",
    startTime=int((datetime.now() - timedelta(hours=1)).timestamp()),
    endTime=int(datetime.now().timestamp()),
    queryString="""
        fields @timestamp, @message
        | filter @message like /ERROR/
        | stats count(*) as errors by bin(5m)
        | sort @timestamp desc
    """,
)
while (r := logs.get_query_results(queryId=q["queryId"]))["status"] not in ("Complete", "Failed"):
    time.sleep(1)

# Act 3 — CloudTrail: who did what?

CloudWatch tells you how the system is behaving. **CloudTrail** tells you what people and services have been doing to it.

Every console click, every CLI command, every SDK call, every service-to-service request — CloudTrail records who made it, what action, when, from where, with what result. It is the audit trail behind every "who deleted that?" question and the foundation of incident investigation.

## AWS CloudTrail

CloudTrail records **every API call** made in your AWS account — console clicks, CLI invocations, SDK calls, service-to-service calls. Every event captures the same five fields:

- **Who** — the IAM principal (user, role, federated identity, or AWS service)
- **What** — the API action: `RunInstances`, `PutObject`, `CreateBucket`, `DeleteTrail`
- **When** — timestamp
- **Where** — source IP address, user agent, region
- **Result** — success or error code, plus the request and response parameters

It's the audit trail for everything that happens on AWS, and the answer to "who did that?" questions.

### Event types

| Type | Captures | Default? | Cost |
|---|---|---|---|
| **Management events** | control-plane operations: create / modify / delete resources, IAM, network changes | yes, free | first copy free |
| **Data events** | data-plane operations: S3 `GetObject`, Lambda `Invoke`, DynamoDB `GetItem` | opt-in | charged |
| **Insights events** | ML-flagged anomalies in API volumes (sudden spikes in `TerminateInstances`, etc.) | opt-in | charged |

Management events are on by default in every account, retained as a queryable **event history** in the console for the **last 90 days** — no setup, free. For longer retention or for data/Insights events, create a **trail** that delivers events to S3 (and optionally a CloudWatch Logs group).

### Trails

A **trail** captures events to S3 with optional delivery to CloudWatch Logs and EventBridge. Two settings worth getting right.

- **Multi-region trail** — captures events from every region into one S3 bucket. The recommended default; without it you only catch the region you create the trail in.
- **Organisation trail** — captures events from every account in an Organisation into a central S3 bucket. The standard pattern for security and audit teams that operate across many accounts.

### CloudTrail Insights

Optional ML-based anomaly detection layered on top of a trail. It baselines the normal call-volume pattern for write APIs (and optionally read APIs) and emits an Insights event when a sudden spike or unusual cessation occurs — "this account just deleted 200 security groups in 5 minutes when it normally deletes none." Insights events drop into the same trail output and can trigger downstream alerting.

### Log file integrity

Enable **log file validation** on a trail and CloudTrail publishes an hourly **SHA-256 digest** of the log files into the same bucket. `aws cloudtrail validate-logs` checks digests against actual log files and detects tampering or deletion. Pair with **S3 Object Lock** on the trail bucket and a restrictive bucket policy and you have a tamper-resistant audit trail.

In [ ]:
import boto3
from datetime import datetime, timedelta

ct = boto3.client("cloudtrail", region_name="us-east-1")

# Multi-region trail with log-file validation, CW Logs delivery, KMS-encrypted output
ct.create_trail(
    Name="org-audit-trail",
    S3BucketName="my-cloudtrail-logs",
    IsMultiRegionTrail=True,
    IncludeGlobalServiceEvents=True,
    EnableLogFileValidation=True,
    CloudWatchLogsLogGroupArn="arn:aws:logs:us-east-1:111122223333:log-group:CloudTrail",
    CloudWatchLogsRoleArn="arn:aws:iam::111122223333:role/CloudTrailCWRole",
    KMSKeyId="alias/cloudtrail-key",
)
ct.start_logging(Name="org-audit-trail")

# Quick lookup against the 90-day event history — "who deleted any S3 bucket this week?"
events = ct.lookup_events(
    LookupAttributes=[{"AttributeKey": "EventName", "AttributeValue": "DeleteBucket"}],
    StartTime=datetime.now() - timedelta(days=7),
    EndTime=datetime.now(),
    MaxResults=10,
)
for e in events["Events"]:
    print(e["EventTime"], e.get("Username"), e.get("Resources"))

# Act 4 — AWS Config: what changed, and is it compliant?

CloudWatch watches behaviour. CloudTrail watches actions. The third question is about state — what does this resource look like, what did it look like at some point in the past, and does it still match the rules?

**AWS Config** records the configuration of every supported AWS resource over time, evaluates each one against rules, and can automatically remediate drift. It is the compliance-posture service that complements CloudTrail's audit log and CloudWatch's performance metrics.

## AWS Config

Config records the **configuration of every supported AWS resource** as it changes, stores that history in S3, and evaluates resources against **rules**. The output is a per-resource timeline — "what did this security group look like at 14:32 last Tuesday, and was it compliant?"

The flow:

1. Config **records** the configuration of EC2 instances, S3 buckets, RDS instances, IAM resources, security groups, VPCs, every supported type — every time something changes.
2. **Configuration history** lands in your S3 bucket; **configuration snapshots** capture point-in-time state.
3. Config **evaluates** each resource against the rules attached to your account.
4. **Notifications** fire on compliance state changes via SNS or EventBridge.
5. Optional **remediation** auto-fixes non-compliant resources via SSM Automation.

### Config Rules

| Type | What it is | Examples |
|---|---|---|
| **AWS managed rules** | 200+ pre-built rules from AWS | `encrypted-volumes`, `s3-bucket-public-read-prohibited`, `restricted-ssh`, `iam-password-policy` |
| **Custom rules** | a Lambda function you write that returns COMPLIANT / NON_COMPLIANT | any organisation-specific check |
| **Service-linked** | rules created by AWS services (Security Hub, Control Tower) | — |

A Config rule re-evaluates **on every configuration change** to the resources it watches (or on a periodic schedule), so non-compliant state surfaces within minutes rather than waiting for a periodic audit.

### Conformance Packs

A **Conformance Pack** is a bundle of Config rules and remediation actions delivered as a single deployable artifact. AWS publishes packs for common frameworks — PCI DSS, HIPAA, NIST 800-53, CIS Benchmarks, AWS Foundational Security Best Practices — and you can author custom packs. Deploy a pack and the whole bundle of rules turns on at once; a compliance report covers all of them together. The right unit when you want "PCI compliance posture" rather than rule-by-rule on/off toggling.

### Aggregator and remediation

- **Aggregator** — pulls Config data from multiple accounts and regions into one view. Required for any cross-account compliance dashboard.
- **Remediation** — attach an SSM Automation document to a Config rule and Config triggers it on non-compliance: enable bucket encryption, revoke an overly permissive SG rule, attach a missing tag. Optional manual approval before execution.

The practical position: **Config is the compliance-posture service**, complementary to CloudTrail's audit log and CloudWatch's performance metrics.

In [ ]:
import boto3

cfg = boto3.client("config", region_name="us-east-1")

# Start the recorder — capture every supported resource type, including IAM (global)
cfg.put_configuration_recorder(ConfigurationRecorder={
    "name": "default",
    "roleARN": "arn:aws:iam::111122223333:role/ConfigRole",
    "recordingGroup": {"allSupported": True, "includeGlobalResourceTypes": True},
})
cfg.put_delivery_channel(DeliveryChannel={
    "name": "default",
    "s3BucketName": "my-config-bucket",
    "snsTopicARN":  "arn:aws:sns:us-east-1:111122223333:config-alerts",
})
cfg.start_configuration_recorder(ConfigurationRecorderName="default")

# Managed rule: flag unencrypted EBS volumes
cfg.put_config_rule(ConfigRule={
    "ConfigRuleName": "encrypted-volumes",
    "Source": {"Owner": "AWS", "SourceIdentifier": "ENCRYPTED_VOLUMES"},
    "Scope":  {"ComplianceResourceTypes": ["AWS::EC2::Volume"]},
})

# Snapshot of current compliance state across all rules
for row in cfg.describe_compliance_by_config_rule()["ComplianceByConfigRules"]:
    print(row["ConfigRuleName"], row["Compliance"]["ComplianceType"])

# Act 5 — Compliance, and picking the right tool

The three observability services answer day-to-day operational questions. Compliance is a different beast — periodic audits, certifications, evidence packages.

Two services answer the compliance side. **AWS Artifact** is the self-serve portal for AWS's own audit reports and the legal agreements you sign on top of them — the HIPAA Business Associate Agreement, the GDPR Data Processing Addendum, the SOC reports. **Audit Manager** continuously collects evidence from your AWS environment and maps it to compliance framework controls for *your* own audits.

The act closes with a side-by-side comparison of CloudWatch, CloudTrail, and Config, and a cue-to-service consolidation across the whole notebook.

## Compliance — Artifact, Audit Manager, Programmes

Two services handle compliance paperwork, and they answer different questions.

### AWS Artifact

Artifact is the self-service portal for AWS's **own** compliance documentation — no support ticket required. Two pieces:

- **Reports** — downloadable: SOC 1/2/3, PCI DSS Attestation of Compliance, ISO 27001 certificate, FedRAMP authorisation packages, GDPR documentation, AWS penetration testing summaries. Hand these to your auditors as evidence of AWS's posture.
- **Agreements** — legal documents you accept on behalf of your account or Organisation: the **Business Associate Agreement (BAA)** for HIPAA, the **GDPR Data Processing Addendum**, and others. Acceptance can be scoped to one account or applied across an Organisation.

The cue: *download AWS's compliance report*, *sign the BAA for HIPAA*, *access the SOC 2* → **Artifact**.

### AWS Audit Manager

Audit Manager helps prepare for *your* audits by **continuously collecting evidence** from your AWS environment and mapping it to compliance framework controls. It pulls in CloudTrail events, Config snapshots, Security Hub findings, and IAM activity, then assembles them as evidence under the relevant framework — PCI DSS, SOC 2, HIPAA, ISO 27001, GDPR, NIST 800-53, custom internal frameworks. The output is an audit-ready report instead of a manual evidence-gathering scramble.

The cue: *prepare for an audit*, *automatically collect evidence*, *map AWS activity to PCI controls* → **Audit Manager**.

### Compliance programmes — what AWS holds

AWS certifies its **infrastructure** against a wide range of standards. Your application may need its own certifications on top, but the underlying platform is already covered:

| Programme | Scope |
|---|---|
| **SOC 1 / 2 / 3** | financial reporting; security & availability; public summary |
| **PCI DSS** | payment card processing |
| **HIPAA** | US healthcare data privacy — AWS is HIPAA-eligible, sign a BAA via Artifact |
| **ISO 27001 / 27017 / 27018** | information security / cloud security / cloud privacy |
| **FedRAMP** | US federal government cloud authorisation |
| **GDPR** | EU data protection — AWS provides the DPA via Artifact |
| **FIPS 140-2** | US cryptographic standard — KMS HSMs are validated |
| **CSA STAR** | Cloud Security Alliance |

The practical reading: AWS being certified doesn't make *you* certified. You still have to (a) use only the services in the relevant scope (e.g. HIPAA-eligible services for HIPAA), (b) configure them correctly (encryption, logging, access control), and (c) sign the relevant agreements via Artifact.

## CloudWatch vs CloudTrail vs Config

The three observability services overlap in casual conversation but answer distinct questions. The map worth memorising:

| Question | Service |
|---|---|
| Is my CPU high? Is my Lambda erroring? Are requests slow? | **CloudWatch** — performance and health |
| Who deleted that S3 bucket? Who changed this IAM policy at 03:14? | **CloudTrail** — API audit log |
| What did my security group look like last Tuesday? Was it compliant? | **Config** — configuration history + rules |
| Alert me when RDS CPU stays > 80% for 10 minutes | CloudWatch alarm on the RDS metric |
| Alert me when someone calls `DeleteTrail` | CloudTrail → CloudWatch metric filter → alarm |
| Alert me when an S3 bucket becomes publicly accessible | Config rule `s3-bucket-public-read-prohibited` → SNS |
| Automatically fix a non-compliant security group | Config rule + SSM Automation remediation |

A mnemonic that works.

- **CloudWatch = how is it doing?** — metrics, logs, alarms, dashboards.
- **CloudTrail = who did what?** — every API call, retained and searchable.
- **Config = what changed, and is it compliant?** — resource state over time, evaluated against rules.

They compose well. CloudTrail delivering to CloudWatch Logs lets you alarm on specific API calls. Config recording an S3 bucket policy change triggers a rule check; if non-compliant, SSM runs a remediation document; the remediation itself shows up in CloudTrail; an alarm on the Config compliance metric pages the team. Each service owns one slice; the answer to most production questions touches at least two.

## Picking the Right Tool

| Cue | Service |
|---|---|
| Whose responsibility is it? | **Shared Responsibility Model** — AWS owns the cloud; you own data, IAM, and configuration |
| Metrics on AWS resources, custom app counters, alarms, dashboards | **CloudWatch** |
| In-OS EC2 memory / disk / application log files | **CloudWatch Agent** |
| Per-container metrics for ECS / EKS | **Container Insights** |
| Function-level cold start / memory / CPU diagnostics | **Lambda Insights** |
| Ad-hoc SQL-like query across log groups | **CloudWatch Logs Insights** |
| Stream logs to OpenSearch / Splunk / Kinesis in real time | **subscription filter** |
| Multi-condition alarm without flap | **composite alarm** |
| Watch the AWS bill | **billing alarm on `EstimatedCharges` in us-east-1** |
| Audit every API call across the account | **CloudTrail** (with a multi-region trail to S3) |
| Track S3 GetObject / Lambda Invoke / DynamoDB reads | **CloudTrail data events** (opt-in) |
| ML-detected anomalies in API volumes | **CloudTrail Insights** |
| Detect trail tampering or deletion | **CloudTrail log file validation** + S3 Object Lock |
| Resource configuration history over time | **AWS Config** |
| Rule-based compliance checking with automatic re-evaluation | **Config Rules** |
| Bundle of rules for PCI / HIPAA / NIST / CIS at once | **Conformance Pack** |
| Auto-fix a non-compliant resource | **Config remediation via SSM Automation** |
| Multi-account compliance dashboard | **Config Aggregator** |
| Download AWS's own SOC / PCI / ISO reports; sign HIPAA BAA | **AWS Artifact** |
| Continuously collect evidence for your own audit | **AWS Audit Manager** |

The shape: CloudWatch watches behaviour, CloudTrail watches actions, Config watches state. Artifact gives you AWS's compliance evidence; Audit Manager builds yours. The Shared Responsibility Model is the framing that decides which of these tools — and which of *your* controls — has to cover any given risk.